
# Multi‑Output Regression for Compost Automation
**Targets:** `temperature_active1..4`  
**Features:** `['timestamp', 'hour', 'phase', 'temperature_active1', 'temperature_active2', 'temperature_active3', 'temperature_active4', 'temperature_curing1', 'temperature_curing2', 'moisture_active1', 'moisture_active2', 'moisture_curing1', 'moisture_curing2', 'oxygen', 'co2', 'methane', 'methane_ppm', 'aeration_on']`

This notebook:
1. Loads and preflights your dataset (`ideal_inflated_data.csv`).
2. Builds **multi‑output regressors** to predict `temperature_active1..4`.
3. Generates **automation signals** (aeration, mixing, phase change) from predictions using configurable rules.
4. Plots results and exports a **control schedule** CSV.


In [ ]:

# --- Imports & Paths ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

# Paths
DATA_PATH = '/mnt/data/ideal_inflated_data.csv'  # change if needed
OUTPUT_DIR = '/mnt/data/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option('display.max_columns', 200)
print('Using data at:', DATA_PATH)


In [ ]:

# --- Load & Preflight ---
df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', list(df.columns))

# Basic coercions
if 'timestamp' in df.columns:
    # Try parse; if already numeric, keep original but also create datetime if possible
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    except Exception:
        # If parse fails, leave as is
        pass

# Expected columns
expected = ['timestamp', 'hour', 'phase', 'temperature_active1', 'temperature_active2', 'temperature_active3',
            'temperature_active4', 'temperature_curing1', 'temperature_curing2',
            'moisture_active1', 'moisture_active2', 'moisture_curing1', 'moisture_curing2',
            'oxygen', 'co2', 'methane', 'methane_ppm', 'aeration_on']

missing = [c for c in expected if c not in df.columns]
extra = [c for c in df.columns if c not in expected]
print('Missing columns:', missing)
print('Extra columns (kept but unused unless added to features):', extra)

# NaN report
nan_summary = df[expected].isna().sum().sort_values(ascending=False) if not missing else df.isna().sum().sort_values(ascending=False)
display(nan_summary.to_frame('NaN_count').head(20))

# Simple imputation for this demo (configurable): forward fill then back fill
df = df.sort_values('timestamp' if 'timestamp' in df.columns else df.index).reset_index(drop=True)
df = df.fillna(method='ffill').fillna(method='bfill')
print('NaNs remaining after simple impute:', int(df.isna().sum().sum()))


In [ ]:

# --- Feature Engineering (Lightweight) ---
# We'll use 'hour' as given, and drop raw 'timestamp' from the model input to avoid leakage/parse issues.
# Encode 'phase' with pandas.get_dummies (compatible with older scikit-learn).

work_df = df.copy()

# Ensure 'aeration_on' is numeric (0/1) if it's boolean/yes-no strings.
if 'aeration_on' in work_df.columns:
    if work_df['aeration_on'].dtype == bool:
        work_df['aeration_on'] = work_df['aeration_on'].astype(int)
    elif work_df['aeration_on'].dtype == object:
        # map common truthy strings to 1
        work_df['aeration_on'] = work_df['aeration_on'].str.lower().map({'1':1,'true':1,'yes':1,'on':1}).fillna(0).astype(int)

# One-hot encode phase (if exists)
if 'phase' in work_df.columns:
    work_df = pd.get_dummies(work_df, columns=['phase'], prefix='phase')

# Define features and targets
target_cols = ['temperature_active1', 'temperature_active2', 'temperature_active3', 'temperature_active4']
feature_cols = ['hour', 'temperature_active1', 'temperature_active2', 'temperature_active3', 'temperature_active4',
                'temperature_curing1', 'temperature_curing2',
                'moisture_active1', 'moisture_active2', 'moisture_curing1', 'moisture_curing2',
                'oxygen', 'co2', 'methane', 'methane_ppm', 'aeration_on']

# Add any phase_ one-hot columns
feature_cols += [c for c in work_df.columns if c.startswith('phase_') and c not in feature_cols]

# Drop rows with missing essential columns (should be none after impute)
keep_cols = (['timestamp'] if 'timestamp' in work_df.columns else []) + list(set(feature_cols + target_cols))
work_df = work_df[keep_cols].copy()

print('Using features:', [c for c in feature_cols])
print('Targets:', target_cols)

# Sort by time if available
if 'timestamp' in work_df.columns:
    work_df = work_df.sort_values('timestamp').reset_index(drop=True)

X = work_df[feature_cols].values
y = work_df[target_cols].values
print('X shape:', X.shape, '| y shape:', y.shape)


In [ ]:

# --- Time-based Train/Test Split ---
# Use last 20% as test to mimic forecasting evaluation.
split_idx = int(0.8 * len(work_df))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
ts_train = work_df['timestamp'].iloc[:split_idx].values if 'timestamp' in work_df.columns else np.arange(split_idx)
ts_test  = work_df['timestamp'].iloc[split_idx:].values if 'timestamp' in work_df.columns else np.arange(len(work_df) - split_idx)

print('Train size:', X_train.shape[0], ' Test size:', X_test.shape[0])


In [ ]:

# --- Utility: One-liner smoothing ---
def smooth(s, w=5):
    """Very simple moving average smoother for 1D arrays or pandas Series."""
    if w <= 1:
        return np.asarray(s)
    return pd.Series(s).rolling(window=w, min_periods=1, center=True).mean().values


In [ ]:

# --- Model A: RandomForestRegressor (native multi-output) ---
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

# Metrics per-target
mae_per_target = {}
r2_per_target = {}
for i, t in enumerate(['TA1','TA2','TA3','TA4']):
    mae = mean_absolute_error(y_test[:, i], pred_rf[:, i])
    r2  = r2_score(y_test[:, i], pred_rf[:, i])
    mae_per_target[t] = mae
    r2_per_target[t] = r2

print('RF MAE per target:', mae_per_target)
print('RF R2 per target:', r2_per_target)

# Aggregate
print('RF Mean MAE:', np.mean(list(mae_per_target.values())))
print('RF Mean R2 :', np.mean(list(r2_per_target.values())))


In [ ]:

# --- Model B: MultiOutputRegressor(Ridge) ---
ridge = MultiOutputRegressor(Ridge(alpha=1.0, random_state=42))
ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)

mae_per_target_r = {}
r2_per_target_r = {}
for i, t in enumerate(['TA1','TA2','TA3','TA4']):
    mae = mean_absolute_error(y_test[:, i], pred_ridge[:, i])
    r2  = r2_score(y_test[:, i], pred_ridge[:, i])
    mae_per_target_r[t] = mae
    r2_per_target_r[t] = r2

print('Ridge MO MAE per target:', mae_per_target_r)
print('Ridge MO R2 per target :', r2_per_target_r)
print('Ridge MO Mean MAE:', np.mean(list(mae_per_target_r.values())))
print('Ridge MO Mean R2 :', np.mean(list(r2_per_target_r.values())))


In [ ]:

# --- Pick the better model (by mean MAE) ---
rf_mean_mae = np.mean(list({k: mean_absolute_error(y_test[:, i], pred_rf[:, i]) for i,k in enumerate(['TA1','TA2','TA3','TA4'])}.values()))
ridge_mean_mae = np.mean(list({k: mean_absolute_error(y_test[:, i], pred_ridge[:, i]) for i,k in enumerate(['TA1','TA2','TA3','TA4'])}.values()))

best_name = 'RandomForest' if rf_mean_mae <= ridge_mean_mae else 'Ridge'
best_pred = pred_rf if best_name == 'RandomForest' else pred_ridge
print('Best model:', best_name)


In [ ]:

# --- Automation Rules (Configurable Thresholds) ---
rules = {
    # Temperature thresholds (°C) – typical compost: keep 55–65°C for sanitization, avoid >70°C for long
    'temp_high': 65.0,        # trigger aeration if exceeded
    'temp_mix_gradient': 8.0, # trigger mixing if max-min across active > this
    'temp_phase_cool': 45.0,  # candidate curing if all active temps < this sustained

    # Gas thresholds (% unless ppm noted)
    'o2_low': 10.0,           # aeration if O2 < 10%
    'co2_high': 15.0,         # aeration if CO2 > 15%
    'ch4_spike_ppm': 1000.0,  # optional alert; not directly used unless you want

    # Moisture (%)
    'moisture_high': 65.0,    # optional aeration/mix if too wet
    'moisture_low': 40.0,     # optional mixing + watering if too dry (not automated here)

    # Debounce windows
    'phase_cool_days': 3      # require N consecutive days/hours (units of your data) under cool threshold
}

def make_control_schedule(df_ref, preds, rules, timestamps):
    """Return a DataFrame with predicted temps and control signals.

    df_ref: original (test) dataframe aligned with preds (rows == len(preds))

    preds: numpy array shape (n, 4) for temperatures active1..4

    rules: dict thresholds

    timestamps: aligned timestamp vector

    """
    out = pd.DataFrame({
        'timestamp': pd.to_datetime(timestamps),
        'pred_TA1': preds[:,0],
        'pred_TA2': preds[:,1],
        'pred_TA3': preds[:,2],
        'pred_TA4': preds[:,3],
        'oxygen': df_ref['oxygen'].values if 'oxygen' in df_ref.columns else np.nan,
        'co2': df_ref['co2'].values if 'co2' in df_ref.columns else np.nan,
        'methane_ppm': df_ref['methane_ppm'].values if 'methane_ppm' in df_ref.columns else np.nan,
        'moisture_active1': df_ref['moisture_active1'].values if 'moisture_active1' in df_ref.columns else np.nan,
        'moisture_active2': df_ref['moisture_active2'].values if 'moisture_active2' in df_ref.columns else np.nan
    })
    # Aeration signal
    ta = out[['pred_TA1','pred_TA2','pred_TA3','pred_TA4']].values
    tmax = ta.max(axis=1)
    tmin = ta.min(axis=1)
    grad = tmax - tmin

    aer_by_temp = (tmax > rules['temp_high'])
    aer_by_o2   = (out['oxygen'] < rules['o2_low']) if 'oxygen' in out.columns else False
    aer_by_co2  = (out['co2'] > rules['co2_high']) if 'co2' in out.columns else False
    aer_by_wet  = ((out['moisture_active1'] > rules['moisture_high']) | (out['moisture_active2'] > rules['moisture_high'])) if 'moisture_active1' in out.columns else False

    out['aeration_recommended'] = (aer_by_temp | aer_by_o2 | aer_by_co2 | aer_by_wet).astype(int)

    # Mixing signal
    mix_by_gradient = (grad > rules['temp_mix_gradient'])
    # Optional: mix if methane spike (indicator of anaerobic pockets)
    mix_by_ch4 = (out['methane_ppm'] > rules['ch4_spike_ppm']) if 'methane_ppm' in out.columns else False
    out['mix_recommended'] = (mix_by_gradient | mix_by_ch4).astype(int)

    # Phase change (active -> curing): all temps below cool threshold for sustained period
    cool = (tmax < rules['temp_phase_cool'])
    # Rolling window 'all below' for N periods
    N = max(int(rules['phase_cool_days']), 1)
    sustained = pd.Series(cool).rolling(window=N, min_periods=N).apply(lambda x: 1.0 if np.all(x) else 0.0).fillna(0).astype(int).values
    out['phase_change_recommended'] = sustained

    return out

# Align a test-slice dataframe for the schedule
test_df_slice = work_df.iloc[int(0.8 * len(work_df)):, :].copy()
schedule = make_control_schedule(test_df_slice, best_pred, rules, ts_test)
display(schedule.head(10))
print('Control schedule rows:', len(schedule))


In [ ]:

# --- Plots: Temps & Controls ---
plt.figure(figsize=(12,5))
plt.plot(schedule['timestamp'], schedule['pred_TA1'], label='pred_TA1')
plt.plot(schedule['timestamp'], schedule['pred_TA2'], label='pred_TA2')
plt.plot(schedule['timestamp'], schedule['pred_TA3'], label='pred_TA3')
plt.plot(schedule['timestamp'], schedule['pred_TA4'], label='pred_TA4')
plt.legend()
plt.title('Predicted Active Temperatures')
plt.xlabel('Time'); plt.ylabel('°C')
plt.show()

plt.figure(figsize=(12,3))
plt.plot(schedule['timestamp'], schedule['aeration_recommended'], label='Aeration=1')
plt.title('Aeration Recommendation')
plt.xlabel('Time'); plt.ylabel('Flag')
plt.yticks([0,1])
plt.show()

plt.figure(figsize=(12,3))
plt.plot(schedule['timestamp'], schedule['mix_recommended'], label='Mix=1')
plt.title('Mixing Recommendation')
plt.xlabel('Time'); plt.ylabel('Flag')
plt.yticks([0,1])
plt.show()

plt.figure(figsize=(12,3))
plt.plot(schedule['timestamp'], schedule['phase_change_recommended'], label='Phase change=1')
plt.title('Phase Change Recommendation (Active→Curing)')
plt.xlabel('Time'); plt.ylabel('Flag')
plt.yticks([0,1])
plt.show()


In [ ]:

# --- Export Control Schedule ---
out_path = os.path.join(OUTPUT_DIR, 'compost_control_schedule.csv')
schedule.to_csv(out_path, index=False)
print('Saved:', out_path)



### (Optional) Keras multi‑output model
If you want a neural network (Keras 2.1.3 / TF 1.5), you can adapt this.  
It uses a **single vector head** for all four temperatures.

```python
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

nn = Sequential()
nn.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
nn.add(Dense(64, activation='relu'))
nn.add(Dense(4))  # 4 temps
nn.compile(optimizer=Adam(lr=1e-3), loss='mae')
nn.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=1)

pred_nn = nn.predict(X_test)
```

Swap `best_pred = pred_nn` above to drive the same automation pipeline.
